# 03 - Distortions (Part 2: introduce distortions, measure degradation)

Applies the 3 locked distortions (salt & pepper noise, motion blur, JPEG
compression) at every configured intensity level, re-runs all 4 tasks, and
plots performance vs. SNR - the PDF's required "range of distortion
intensities, measure as SNR" evaluation.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in str(get_ipython())
REPO_URL = "https://github.com/Shir-Siman-Tov/Image-Processing-Project.git"
REPO_DIR = "/content/Image-Processing-Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        # Runtime already had this repo cloned from an earlier cell run in this
        # session - pull so we don't keep running against a stale checkout.
        get_ipython().system(f"git -C {REPO_DIR} pull")
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q -e .")
    get_ipython().system("pip install -q -r requirements.txt")

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

from ipproj import config
from ipproj.datasets import kitti, kitti_flow
from ipproj.datasets.kitti import read_image
from ipproj.datasets.kitti_flow import read_kitti_flow_png
from ipproj.datasets.materialize import materialize_transformed
from ipproj.tasks import feature_matching, optical_flow, object_detection, semantic_segmentation
from ipproj.distortions import REGISTRY as DISTORTIONS
from ipproj.metrics.snr import compute_snr_db
from ipproj.viz.plotting import (
    plot_before_after, plot_grouped_bar_per_class, plot_image_grid, plot_metric_bars_by_condition,
    plot_metric_vs_intensity, plot_metric_vs_snr_by_distortion, save_figure,
)

detection_splits = kitti.load_object_detection_subset()
segmentation_splits = kitti.load_semantic_segmentation_subset()
flow_splits = kitti_flow.load_optical_flow_subset()

checkpoint_path = (config.CHECKPOINT_ROOT / "yolo_clean_baseline_path.txt").read_text().strip()
yolo_model = YOLO(checkpoint_path)
segformer_model, segformer_processor = semantic_segmentation.load_pretrained()

clean_baseline = pd.read_csv(config.RESULTS_ROOT / "02_clean_baseline.csv").set_index(["task", "metric"])["value"]

## Before/after visualization per distortion (strongest intensity)

In [ ]:
sample_image = read_image(detection_splits["test"][0].image_path)
for name, module in DISTORTIONS.items():
    distorted = module.distort(sample_image, level=len(module.LEVELS) - 1)
    fig = plot_before_after(sample_image, distorted, title_after=f"{name} (max intensity)")
    save_figure(fig, f"03_distortions/before_after_{name}.png")

## Distortion severity grid (all distortions x all levels)

One figure showing every distortion (row) at every configured severity level
(column) applied to the same sample image, so the progression of each
distortion family is visible at a glance alongside the others.

In [ ]:
grid_images = []
grid_titles = []
for name, module in DISTORTIONS.items():
    for level in range(len(module.LEVELS)):
        grid_images.append(module.distort(sample_image, level))
        grid_titles.append(f"{name}\nlevel {level + 1}: {module.LEVELS[level]}")

n_levels = len(next(iter(DISTORTIONS.values())).LEVELS)
fig = plot_image_grid(grid_images, titles=grid_titles, ncols=n_levels)
fig.suptitle("All distortions x all severity levels")
save_figure(fig, "03_distortions/all_distortions_all_levels_grid.png")

## Per-class breakdown: clean vs. a severe distortion

Detection mAP and segmentation IoU per class, clean vs. the most severe
level of one representative distortion (motion blur, level 5/5), to see
which classes break first under a harsh condition rather than only looking
at aggregate metrics.

In [ ]:
SEVERE_DISTORTION = "motion_blur"
SEVERE_LEVEL = len(DISTORTIONS[SEVERE_DISTORTION].LEVELS) - 1
severe_module = DISTORTIONS[SEVERE_DISTORTION]
severe_distort_fn = lambda img, m=severe_module, l=SEVERE_LEVEL: m.distort(img, l)
severe_out_dir = config.DISTORTED_ROOT / SEVERE_DISTORTION / f"level_{SEVERE_LEVEL}"
severe_label = f"{SEVERE_DISTORTION} level {SEVERE_LEVEL + 1}"

# 02_clean_baseline.csv only stores aggregate scalars, so the clean per-class
# breakdown is recomputed here to pair against the severe-distortion one.
clean_detection_metrics = object_detection.evaluate(yolo_model, detection_splits["test"])
clean_segmentation_iou = semantic_segmentation.evaluate(segformer_model, segformer_processor, segmentation_splits["test"])

severe_detection = materialize_transformed(detection_splits["test"], severe_distort_fn, severe_out_dir / "detection")
severe_detection_metrics = object_detection.evaluate(yolo_model, severe_detection)

severe_segmentation = materialize_transformed(segmentation_splits["test"], severe_distort_fn, severe_out_dir / "segmentation")
severe_segmentation_iou = semantic_segmentation.evaluate(segformer_model, segformer_processor, severe_segmentation)

def detection_per_class_dict(metrics):
    # torchmetrics only reports classes present in that call's ground truth,
    # so align by class name rather than assuming identical order/coverage
    # between the clean and severe evaluate() calls.
    return {
        config.KITTI_DETECTION_CLASSES[i]: v
        for i, v in zip(metrics["classes"].tolist(), metrics["map_per_class"].tolist())
    }

clean_detection_by_class = detection_per_class_dict(clean_detection_metrics)
severe_detection_by_class = detection_per_class_dict(severe_detection_metrics)

detection_series = {
    "clean": [clean_detection_by_class.get(c, 0.0) for c in config.KITTI_DETECTION_CLASSES],
    severe_label: [severe_detection_by_class.get(c, 0.0) for c in config.KITTI_DETECTION_CLASSES],
}
fig = plot_grouped_bar_per_class(
    config.KITTI_DETECTION_CLASSES, detection_series, ylabel="mAP",
    title=f"Detection mAP per class: clean vs. {SEVERE_DISTORTION} "
          f"(level {SEVERE_LEVEL + 1}/{len(severe_module.LEVELS)}, k={severe_module.LEVELS[SEVERE_LEVEL]})",
)
save_figure(fig, "03_distortions/detection_map_per_class_clean_vs_motion_blur_severe.png")

segmentation_series = {
    "clean": clean_segmentation_iou.tolist(),
    severe_label: severe_segmentation_iou.tolist(),
}
fig = plot_grouped_bar_per_class(
    config.CITYSCAPES_TRAINID_LABELS, segmentation_series, ylabel="IoU",
    title=f"Segmentation IoU per class: clean vs. {SEVERE_DISTORTION} "
          f"(level {SEVERE_LEVEL + 1}/{len(severe_module.LEVELS)}, k={severe_module.LEVELS[SEVERE_LEVEL]})",
)
save_figure(fig, "03_distortions/segmentation_iou_per_class_clean_vs_motion_blur_severe.png")

## Degradation sweep

For each distortion x intensity level: distort the test split, re-run all 4
tasks, and record metrics + a representative SNR value. Distorted images are
persisted under `config.DISTORTED_ROOT` (via `materialize_transformed`) so
`tasks.*.evaluate()` can be reused unchanged - it always reads from a
sample's `.image_path`.

In [ ]:
results = []

for name, module in DISTORTIONS.items():
    for level in range(len(module.LEVELS)):
        distort_fn = lambda img, m=module, l=level: m.distort(img, l)
        out_dir = config.DISTORTED_ROOT / name / f"level_{level}"

        distorted_detection = materialize_transformed(detection_splits["test"], distort_fn, out_dir / "detection")
        detection_metrics = object_detection.evaluate(yolo_model, distorted_detection)

        distorted_segmentation = materialize_transformed(segmentation_splits["test"], distort_fn, out_dir / "segmentation")
        segmentation_metrics = semantic_segmentation.evaluate(segformer_model, segformer_processor, distorted_segmentation)

        match_accuracies = []
        good_match_ratios = []
        orb_snr_values = []
        for sample in detection_splits["test"][:20]:
            clean = read_image(sample.image_path)
            distorted = module.distort(clean, level)
            _, _, _, accuracy, good_match_ratio = feature_matching.match(clean, distorted)
            match_accuracies.append(accuracy)
            good_match_ratios.append(good_match_ratio)
            orb_snr_values.append(compute_snr_db(clean, distorted))

        epe_values = []
        fl_error_values = []
        flow_snr_values = []
        for sample in flow_splits["test"]:
            frame1 = read_image(sample.frame1_path)
            frame2 = read_image(sample.frame2_path)
            distorted_frame2 = module.distort(frame2, level)
            gt_flow, valid = read_kitti_flow_png(sample.flow_gt_path)
            flow_metrics = optical_flow.evaluate(frame1, distorted_frame2, gt_flow, valid)
            epe_values.append(flow_metrics["epe"])
            fl_error_values.append(flow_metrics["fl_error"])
            flow_snr_values.append(compute_snr_db(frame2, distorted_frame2))

        detection_snr_values = []
        for sample in detection_splits["test"]:
            clean = read_image(sample.image_path)
            detection_snr_values.append(compute_snr_db(clean, module.distort(clean, level)))

        segmentation_snr_values = []
        for sample in segmentation_splits["test"]:
            clean = read_image(sample.image_path)
            segmentation_snr_values.append(compute_snr_db(clean, module.distort(clean, level)))

        results.append({
            "distortion": name,
            "level": level,
            "orb_snr_db": float(np.mean(orb_snr_values)),
            "flow_snr_db": float(np.mean(flow_snr_values)),
            "detection_snr_db": float(np.mean(detection_snr_values)),
            "segmentation_snr_db": float(np.mean(segmentation_snr_values)),
            "match_accuracy": float(np.mean(match_accuracies)),
            "good_match_ratio": float(np.mean(good_match_ratios)),
            "epe": float(np.mean(epe_values)),
            "fl_error": float(np.mean(fl_error_values)),
            "map": float(detection_metrics["map"]),
            "map_50": float(detection_metrics["map_50"]),
            "map_75": float(detection_metrics["map_75"]),
            "mar_100": float(detection_metrics["mar_100"]),
            "mean_iou": float(segmentation_metrics.mean()),
        })

distortion_results = pd.DataFrame(results)
config.RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
distortion_results.to_csv(config.RESULTS_ROOT / "03_distortions.csv", index=False)
distortion_results

## Performance vs. SNR

In [ ]:
for name in DISTORTIONS:
    subset = distortion_results[distortion_results["distortion"] == name].sort_values("detection_snr_db")
    fig = plot_metric_vs_intensity(
        subset["detection_snr_db"].tolist(),
        {
            "mAP": subset["map"].tolist(),
            "mAP@0.5": subset["map_50"].tolist(),
            "mean IoU": subset["mean_iou"].tolist(),
            "match accuracy": subset["match_accuracy"].tolist(),
        },
        xlabel="SNR (dB)", ylabel="metric value", title=f"Performance vs SNR - {name}",
    )
    save_figure(fig, f"03_distortions/performance_vs_snr_{name}.png")

## Performance vs. mean SNR, per task (with clean-baseline reference)

One figure per task metric: a curve per distortion type (own mean SNR per
level, averaged over that task's own sample images) plus a horizontal
reference line at the clean (undistorted) baseline value from
`02_clean_baseline.csv`. ORB's reference line is a clean-vs-clean self-match
accuracy (a trivial/ceiling case, not a clean-vs-distorted comparison like
the other tasks' baselines) - see `02_clean_baseline.ipynb`.

In [ ]:
metric_plots = [
    ("match_accuracy", "orb_snr_db", "match accuracy", ("feature_matching", "match_accuracy"), "feature_matching_accuracy_vs_snr"),
    ("epe", "flow_snr_db", "EPE", ("optical_flow", "epe"), "optical_flow_epe_vs_snr"),
    ("fl_error", "flow_snr_db", "Fl-error", ("optical_flow", "fl_error"), "optical_flow_fl_error_vs_snr"),
    ("map", "detection_snr_db", "mAP", ("object_detection", "map"), "object_detection_map_vs_snr"),
    ("mean_iou", "segmentation_snr_db", "mean IoU", ("semantic_segmentation", "mean_iou"), "semantic_segmentation_iou_vs_snr"),
]

for metric_col, snr_col, ylabel, baseline_key, filename in metric_plots:
    snr_by_distortion = {}
    metric_by_distortion = {}
    for name in DISTORTIONS:
        subset = distortion_results[distortion_results["distortion"] == name]
        snr_by_distortion[name] = subset[snr_col].tolist()
        metric_by_distortion[name] = subset[metric_col].tolist()

    clean_reference = float(clean_baseline.loc[baseline_key]) if baseline_key is not None else None

    fig = plot_metric_vs_snr_by_distortion(
        snr_by_distortion, metric_by_distortion, clean_reference,
        xlabel="mean SNR (dB)", ylabel=ylabel, title=f"{ylabel} vs mean SNR",
    )
    save_figure(fig, f"03_distortions/{filename}.png")

## Robustness bar charts (metric vs. distortion severity)

One bar chart per task: a bar per condition ("clean" + every distortion x
severity level, 5 levels each), colored by distortion family, with a red
dashed line at the clean baseline from `02_clean_baseline.csv`. Unlike the
SNR-axis plots above, the x-axis here is the actual severity parameter
(noise fraction, blur kernel size, JPEG quality), so severity within a
distortion family is directly readable. Optical flow's EPE is an error
metric (lower is better) - taller bars mean worse performance, opposite of
the other three charts.

In [ ]:
CLEAN_BAR_COLOR = "#c3c2b7"
DISTORTION_FAMILY_COLORS = {
    "salt_pepper": "#2a78d6",
    "motion_blur": "#eb6834",
    "jpeg_compression": "#1baf7a",
}
SEVERITY_LABEL_FNS = {
    "salt_pepper": lambda v: f"{v:.0%}",
    "motion_blur": lambda v: f"k={v}",
    "jpeg_compression": lambda v: f"q={v}",
}

condition_labels = ["clean"]
condition_colors = [CLEAN_BAR_COLOR]
for name, module in DISTORTIONS.items():
    for raw_value in module.LEVELS:
        condition_labels.append(f"{name}\n{SEVERITY_LABEL_FNS[name](raw_value)}")
        condition_colors.append(DISTORTION_FAMILY_COLORS[name])

In [ ]:
robustness_bar_plots = [
    ("match_accuracy", ("feature_matching", "match_accuracy"), "match accuracy", "ORB feature matching robustness"),
    ("epe", ("optical_flow", "epe"), "EPE (lower is better)", "Optical flow (Farneback) robustness"),
    ("map", ("object_detection", "map"), "mAP@0.5:0.95", "YOLOv8 object detection robustness"),
    ("mean_iou", ("semantic_segmentation", "mean_iou"), "mean IoU", "SegFormer semantic segmentation robustness"),
]

for metric_col, baseline_key, ylabel, title in robustness_bar_plots:
    clean_reference = float(clean_baseline.loc[baseline_key])
    values = [clean_reference]
    for name, module in DISTORTIONS.items():
        for level in range(len(module.LEVELS)):
            row = distortion_results[(distortion_results["distortion"] == name) & (distortion_results["level"] == level)]
            values.append(float(row[metric_col].iloc[0]))

    fig = plot_metric_bars_by_condition(
        condition_labels, values, condition_colors, ylabel=ylabel,
        title=f"{title}\n{ylabel} vs Distortion Severity",
        reference_value=clean_reference, reference_label="clean baseline",
    )
    save_figure(fig, f"03_distortions/{metric_col}_robustness_bars.png")